In [2]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import sys

# Packages are loaded via PYSPARK_SUBMIT_ARGS set in compose.yml.
# pyspark-notebook:spark-44.0.1 — print spark.version to confirm (must stay on Spark 4.0.x for Iceberg 4.0 runtime).

spark = (
    SparkSession.builder
    .appName("project2")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")

    # ── Iceberg ──────────────────────────────────────────────────────────────
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    # Catalog named 'lakehouse' — use it as: lakehouse.<database>.<table>
    .config("spark.sql.catalog.lakehouse",
            "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.lakehouse.type",      "rest")
    .config("spark.sql.catalog.lakehouse.uri",       "http://iceberg-rest:8181")
    .config("spark.sql.catalog.lakehouse.warehouse", "s3://warehouse/")
    # S3FileIO writes data files directly to MinIO
    .config("spark.sql.catalog.lakehouse.io-impl",
            "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.lakehouse.s3.endpoint",          "http://minio:9000")
    .config("spark.sql.catalog.lakehouse.s3.path-style-access", "true")
    .config("spark.sql.catalog.lakehouse.s3.access-key-id",     os.environ["AWS_ACCESS_KEY_ID"])
    .config("spark.sql.catalog.lakehouse.s3.secret-access-key", os.environ["AWS_SECRET_ACCESS_KEY"])
    .config("spark.sql.catalog.lakehouse.s3.region", "us-east-1")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version}   catalog: lakehouse")

# ── Create your database once ──────────────────────────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS lakehouse.taxi")
print("S3A implementation loaded:", spark._jsc.hadoopConfiguration().get("fs.s3a.impl"))

Spark 4.0.1   catalog: lakehouse
S3A implementation loaded: org.apache.hadoop.fs.s3a.S3AFileSystem


In [3]:
BOOTSTRAP = "kafka:9092"
TOPIC = "taxi-trips"

raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", BOOTSTRAP)
    .option("subscribe", TOPIC)
    .option("startingOffsets", "earliest")
    .load()
)

In [4]:
def check_class(class_name):
    try:
        spark._jvm.java.lang.Class.forName(class_name)
        print(f"✅ FOUND: {class_name}")
    except Exception as e:
        print(f"❌ MISSING: {class_name}")

print("Checking Hadoop-AWS Internals:")
check_class("org.apache.hadoop.fs.s3a.S3AFileSystem") # The main driver
check_class("org.apache.hadoop.fs.s3a.S3A")          # The checkpoint delegate
check_class("com.amazonaws.services.s3.AmazonS3")    # The SDK Bundle

Checking Hadoop-AWS Internals:
❌ MISSING: org.apache.hadoop.fs.s3a.S3AFileSystem
❌ MISSING: org.apache.hadoop.fs.s3a.S3A
❌ MISSING: com.amazonaws.services.s3.AmazonS3


In [5]:
BRONZE_TABLE = "lakehouse.taxi.bronze_raw_events"
CHECKPOINT_PATH = "/home/jovyan/work/checkpoints/bronze_raw_events"

# Create a raw Bronze table that mirrors Kafka payload + metadata as-is.
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
    key BINARY,
    value BINARY,
    topic STRING,
    partition INT,
    offset BIGINT,
    timestamp TIMESTAMP,
    timestampType INT
) USING iceberg
""")

bronze_query = (
    raw_stream.writeStream
    .format("iceberg")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_PATH)
    .toTable(BRONZE_TABLE)
)

print(f"Bronze stream started: {bronze_query.id}")
print(f"Checkpoint path: {CHECKPOINT_PATH}")

Bronze stream started: 4b39f6b9-a0c0-48f5-a13c-f859dfd922ae
Checkpoint path: /home/jovyan/work/checkpoints/bronze_raw_events


In [6]:
raw = spark.read.table("lakehouse.taxi.bronze_raw_events")

raw.createOrReplaceTempView("bronze")

spark.sql("SELECT count(*) FROM bronze").show()
spark.sql("SELECT * FROM bronze LIMIT 3").show()

CHECKPOINT_PATH = "/home/jovyan/work/checkpoints/bronze_raw_events"

+--------+
|count(1)|
+--------+
|   46035|
+--------+

+----+--------------------+----------+---------+------+--------------------+-------------+
| key|               value|     topic|partition|offset|           timestamp|timestampType|
+----+--------------------+----------+---------+------+--------------------+-------------+
|[31]|[7B 22 56 65 6E 6...|taxi-trips|        0|  4000|2026-04-04 14:08:...|            0|
|[31]|[7B 22 56 65 6E 6...|taxi-trips|        0|  6214|2026-04-04 14:42:...|            0|
|[31]|[7B 22 56 65 6E 6...|taxi-trips|        0|  6215|2026-04-04 14:42:...|            0|
+----+--------------------+----------+---------+------+--------------------+-------------+



In [7]:
## SILVER

In [8]:
from pyspark.sql.types import (
    StructType, StructField,
    LongType, DoubleType, StringType, IntegerType, TimestampType
)

In [9]:
## Define schema

In [10]:
TRIP_SCHEMA = StructType([
    StructField("VendorID",              LongType(),   True),
    StructField("tpep_pickup_datetime",  StringType(), True),   
    StructField("tpep_dropoff_datetime", StringType(), True),   
    StructField("passenger_count",       DoubleType(), True),   
    StructField("trip_distance",         DoubleType(), True),
    StructField("RatecodeID",            DoubleType(), True),
    StructField("store_and_fwd_flag",    StringType(), True),
    StructField("PULocationID",          LongType(),   True),
    StructField("DOLocationID",          LongType(),   True),
    StructField("payment_type",          LongType(),   True),
    StructField("fare_amount",           DoubleType(), True),
    StructField("extra",                 DoubleType(), True),
    StructField("mta_tax",               DoubleType(), True),
    StructField("tip_amount",            DoubleType(), True),
    StructField("tolls_amount",          DoubleType(), True),
    StructField("improvement_surcharge", DoubleType(), True),
    StructField("total_amount",          DoubleType(), True),
    StructField("congestion_surcharge",  DoubleType(), True),
    StructField("Airport_fee",           DoubleType(), True),
    StructField("cbd_congestion_fee",    DoubleType(), True),
])

In [11]:
## 2. Read bronze — batch

In [12]:
silver_batch = (
     spark.read
          .format("iceberg")                        
          .table("lakehouse.taxi.bronze_raw_events")
)


In [ ]:

def parse(silver_read):
    # ---------------------------------------------------------------------------
    # 3. R1 — Parse JSON payload
    # ---------------------------------------------------------------------------
    parsed = (
        silver_read
        .withColumn("value_str", F.col("value").cast("string"))
        .withColumn("trip",      F.from_json(F.col("value_str"), TRIP_SCHEMA))
        .select("trip.*",
                F.col("timestamp").alias("kafka_ingest_ts"))   # keep Kafka metadata
    )
    
    
    # ---------------------------------------------------------------------------
    # 4. R2 — Cast timestamp strings → TimestampType
    #    Input format: "2025-01-01T12:45:51" (ISO-8601 without timezone)
    # ---------------------------------------------------------------------------
    parsed = (
        parsed
        .withColumn("pickup_datetime",
                    F.to_timestamp("tpep_pickup_datetime",  "yyyy-MM-dd'T'HH:mm:ss"))
        .withColumn("dropoff_datetime",
                    F.to_timestamp("tpep_dropoff_datetime", "yyyy-MM-dd'T'HH:mm:ss"))
        .drop("tpep_pickup_datetime", "tpep_dropoff_datetime")
    )
    return parsed


In [14]:
## Clean

In [15]:
def clean(parsed):
    cleaned = (
        parsed
    
        # R3 — passenger_count: float → int; must be 1–9
        .withColumn("passenger_count",
                    F.when(
                        F.col("passenger_count").cast(IntegerType()).between(1, 9),
                        F.col("passenger_count").cast(IntegerType())
                    ).otherwise(F.lit(None).cast(IntegerType())))
    
        # R4 — trip_distance must be positive
        .withColumn("trip_distance",
                    F.when(F.col("trip_distance") > 0, F.col("trip_distance"))
                     .otherwise(None))
    
        # R5 — RatecodeID valid domain: 1=Standard, 2=JFK, 3=Newark,
        #        4=Nassau/Westchester, 5=Negotiated, 6=Group ride
        .withColumn("RatecodeID",
                    F.when(F.col("RatecodeID").cast(IntegerType()).between(1, 6),
                           F.col("RatecodeID").cast(IntegerType()))
                     .otherwise(None))
    
        # R6 — payment_type valid domain: 1=Credit, 2=Cash, 3=No charge,
        #        4=Dispute, 5=Unknown, 6=Voided
        .withColumn("payment_type",
                    F.when(F.col("payment_type").between(1, 6), F.col("payment_type"))
                     .otherwise(None))
    
        # R7 — fare_amount cannot be negative
        .withColumn("fare_amount",
                    F.when(F.col("fare_amount") >= 0, F.col("fare_amount"))
                     .otherwise(None))
    
        # R8 — total_amount cannot be negative
        .withColumn("total_amount",
                    F.when(F.col("total_amount") >= 0, F.col("total_amount"))
                     .otherwise(None))
    
        # R9 — dropoff must be strictly after pickup
        .withColumn("is_valid_window",
                    F.col("dropoff_datetime") > F.col("pickup_datetime"))
        .filter(F.col("is_valid_window"))
        .drop("is_valid_window")
    
        # Derived: trip duration in minutes (convenient for analytics)
        .withColumn("trip_duration_min",
                    F.round(
                        (F.unix_timestamp("dropoff_datetime")
                         - F.unix_timestamp("pickup_datetime")) / 60.0,
                        2))
    
        # Partition column for Iceberg — date extracted from pickup
        .withColumn("pickup_date", F.to_date("pickup_datetime"))
    )
    return cleaned


In [16]:
## Deduplicate

In [17]:
def dedup(cleaned):
    DEDUP_KEY = ["VendorID", "pickup_datetime", "dropoff_datetime",
                 "PULocationID", "DOLocationID"]
    return cleaned.dropDuplicates(DEDUP_KEY)


In [18]:
## Zone enrichment (pickup + dropoff)

In [19]:
def enrich(deduped):
    zones = spark.read.parquet("/home/jovyan/project/data/taxi_zone_lookup.parquet").alias("z")
    
    enriched = (
        deduped.alias("t")
    
        # Pickup zone
        .join(
            zones.select(
                F.col("LocationID").alias("pu_loc_id"),
                F.col("Zone").alias("pickup_zone"),
                F.col("Borough").alias("pickup_borough"), 
                F.col("service_zone").alias("pickup_service_zone"),
            ),
            F.col("t.PULocationID") == F.col("pu_loc_id"),
            how="left"
        )
        .drop("pu_loc_id")
    
        # Dropoff zone
        .join(
            zones.select(
                F.col("LocationID").alias("do_loc_id"),
                F.col("Zone").alias("dropoff_zone"),
                F.col("Borough").alias("dropoff_borough"),
                F.col("service_zone").alias("dropoff_service_zone"),
            ),
            F.col("t.DOLocationID") == F.col("do_loc_id"),
            how="left"
        )
        .drop("do_loc_id")
    )
    return enriched
 


In [20]:
## Final column order

In [21]:
def order(enriched):
    silver = enriched.select(
        # identifiers
        "VendorID",
        # time
        "pickup_datetime", "dropoff_datetime", "trip_duration_min", "pickup_date",
        # geography
        "PULocationID", "pickup_zone",  "pickup_borough",
        "DOLocationID", "dropoff_zone", "dropoff_borough",
        # trip facts
        "passenger_count", "trip_distance", "RatecodeID", "store_and_fwd_flag",
        # fares
        "fare_amount", "extra", "mta_tax", "tip_amount",
        "tolls_amount", "improvement_surcharge",
        "congestion_surcharge", "Airport_fee", "cbd_congestion_fee",
        "total_amount", "payment_type", 
        # lineage
        "kafka_ingest_ts",
    )
    return silver

def transform(silver_stream):
    
    return order(enrich(dedup(clean(parse(silver_stream)))))


In [22]:
## Write to silver table


silver_schema = transform(silver_batch).schema
print("--"*30)

silver_schema

------------------------------------------------------------


StructType([StructField('VendorID', LongType(), True), StructField('pickup_datetime', TimestampType(), True), StructField('dropoff_datetime', TimestampType(), True), StructField('trip_duration_min', DoubleType(), True), StructField('pickup_date', DateType(), True), StructField('PULocationID', LongType(), True), StructField('pickup_zone', StringType(), True), StructField('pickup_borough', StringType(), True), StructField('DOLocationID', LongType(), True), StructField('dropoff_zone', StringType(), True), StructField('dropoff_borough', StringType(), True), StructField('passenger_count', IntegerType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', IntegerType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), Stru

In [23]:
SILVER_CHECKPOINT = "/home/jovyan/work/checkpoints/bronze_raw_events"

silver_schema = transform(silver_batch).schema
table_name = "lakehouse.taxi.silver_trips"

if not spark.catalog.tableExists(table_name):
    print(f"Creating {table_name} for the first time...")
    
    # Create empty table from schema (Metadata only - no streaming conflict)
    spark.catalog.createTable(table_name, schema=silver_schema, source="iceberg")
    
    # Apply partitioning and optimizations via SQL
    spark.sql(f"ALTER TABLE {table_name} ADD PARTITION FIELD pickup_date")
    spark.sql(f"ALTER TABLE {table_name} SET TBLPROPERTIES ('write.format.default'='parquet', 'write.parquet.compression-codec'='zstd')")
    print("Table initialized.")
else:
    print(f"{table_name} already exists. Skipping initialization.")
silver_batch_transformed = transform(silver_batch)

(
    silver_batch_transformed
    .writeTo("lakehouse.taxi.silver_trips")
    .overwritePartitions()
)

spark.read.table("lakehouse.taxi.silver_trips").count()


lakehouse.taxi.silver_trips already exists. Skipping initialization.


45812

In [24]:
spark.read.table("lakehouse.taxi.silver_trips").count()

45812

**Gold**

Reading batch from the silver layer

In [25]:
gold_batch = (
     spark.read
          .format("iceberg")                        
          .table("lakehouse.taxi.silver_trips")
)

def gold_writer(df, table_name):
    return (
        df.writeTo(table_name)
        .using("iceberg")
        .tableProperty("write.format.default", "parquet")
        .tableProperty("write.parquet.compression-codec", "zstd")
    )

Aggregation 1: Trip counts per pickup hour

In [26]:
def get_trip_counts_per_pickup_hour(batch):
    pick_up_hour_df = batch.withColumn(
        "pickup_hour", F.date_trunc("hour", F.col("pickup_datetime"))
    )

    df = pick_up_hour_df.groupBy(
        "pickup_date", "pickup_hour"
        ).count().withColumnRenamed("count", "trip_count")
    
    return df

In [27]:
gold_trips_per_hour_table = "lakehouse.taxi.gold_trips_per_hour"

gold_trips_per_hour_df = get_trip_counts_per_pickup_hour(gold_batch)

if not spark.catalog.tableExists(gold_trips_per_hour_table):
    # First time: define layout + load data in one step
    gold_writer(gold_trips_per_hour_df, gold_trips_per_hour_table).partitionedBy("pickup_date").create()
else:
    # Every batch: replace only partitions present in gold_df
    gold_writer(gold_trips_per_hour_df, gold_trips_per_hour_table).overwritePartitions()

In [28]:
gold_trips_per_hour_after_ins = spark.table(gold_trips_per_hour_table)
gold_trips_per_hour_after_ins.show(20)

+-----------+-------------------+----------+
|pickup_date|        pickup_hour|trip_count|
+-----------+-------------------+----------+
| 2024-12-31|2024-12-31 23:00:00|        15|
| 2024-12-31|2024-12-31 20:00:00|         3|
| 2024-12-31|2024-12-31 21:00:00|         3|
| 2025-01-01|2025-01-01 13:00:00|      4041|
| 2025-01-01|2025-01-01 03:00:00|      3290|
| 2025-01-01|2025-01-01 08:00:00|       921|
| 2025-01-01|2025-01-01 10:00:00|      1974|
| 2025-01-01|2025-01-01 01:00:00|      5663|
| 2025-01-01|2025-01-01 00:00:00|      5795|
| 2025-01-01|2025-01-01 09:00:00|      1286|
| 2025-01-01|2025-01-01 12:00:00|      3445|
| 2025-01-01|2025-01-01 04:00:00|      1994|
| 2025-01-01|2025-01-01 07:00:00|       820|
| 2025-01-01|2025-01-01 15:00:00|      3056|
| 2025-01-01|2025-01-01 05:00:00|       884|
| 2025-01-01|2025-01-01 02:00:00|      4738|
| 2025-01-01|2025-01-01 06:00:00|       800|
| 2025-01-01|2025-01-01 14:00:00|      4379|
| 2025-01-01|2025-01-01 11:00:00|      2704|
| 2025-01-

Aggregation 2: Average fare per pickup zone

In [29]:
def get_average_fare_per_pickup_zone(batch):
    df = batch.groupBy(
        "pickup_date", "PULocationID", "pickup_borough", "pickup_zone"
        ).agg(
            F.round(F.avg("fare_amount"), 2).alias("average_fare")
        ).withColumnRenamed(
            "PULocationID", "pickup_location_id"
        )
    
    return df

In [30]:
gold_average_fare_per_zone_table = "lakehouse.taxi.gold_average_fare_per_zone"

gold_average_fare_per_zone_df = get_average_fare_per_pickup_zone(gold_batch)

if not spark.catalog.tableExists(gold_average_fare_per_zone_table):
    gold_writer(gold_average_fare_per_zone_df, gold_average_fare_per_zone_table).partitionedBy("pickup_date").create()
else:
    gold_writer(gold_average_fare_per_zone_df, gold_average_fare_per_zone_table).overwritePartitions()

In [31]:
gold_average_fare_per_zone_after_ins = spark.table("lakehouse.taxi.gold_average_fare_per_zone")
gold_average_fare_per_zone_after_ins.show(20)

+-----------+------------------+--------------+--------------------+------------------+
|pickup_date|pickup_location_id|pickup_borough|         pickup_zone|      average_fare|
+-----------+------------------+--------------+--------------------+------------------+
| 2024-12-31|                48|     Manhattan|        Clinton East|               9.3|
| 2024-12-31|               114|     Manhattan|Greenwich Village...|              21.2|
| 2024-12-31|               186|     Manhattan|Penn Station/Madi...|              16.3|
| 2024-12-31|               249|     Manhattan|        West Village|              23.3|
| 2024-12-31|                68|     Manhattan|        East Chelsea|              14.9|
| 2024-12-31|               142|     Manhattan| Lincoln Square East|              12.1|
| 2024-12-31|                42|     Manhattan|Central Harlem North|              16.3|
| 2024-12-31|               179|        Queens|         Old Astoria|               7.9|
| 2024-12-31|               163|

Aggregation 3: Revenue per pick up zone

In [32]:
def get_revenue_per_pickup_zone(batch):
    df = batch.groupBy(
        "pickup_date", "PULocationID", "pickup_borough", "pickup_zone"
    ).agg(
        F.round(F.sum("total_amount"), 2).alias("total_revenue"),
    ).withColumnRenamed("PULocationID", "pickup_location_id")
    return df

In [33]:
gold_revenue_per_zone_table = "lakehouse.taxi.gold_revenue_per_zone"

gold_revenue_per_zone_df = get_revenue_per_pickup_zone(gold_batch)

if not spark.catalog.tableExists(gold_revenue_per_zone_table):
    gold_writer(gold_revenue_per_zone_df, gold_revenue_per_zone_table).partitionedBy("pickup_date").create()
else:
    gold_writer(gold_revenue_per_zone_df, gold_revenue_per_zone_table).overwritePartitions()

In [34]:
gold_revenue_per_zone_after_ins = spark.table("lakehouse.taxi.gold_revenue_per_zone")
gold_revenue_per_zone_after_ins.show(20)

+-----------+------------------+--------------+--------------------+------------------+
|pickup_date|pickup_location_id|pickup_borough|         pickup_zone|     total_revenue|
+-----------+------------------+--------------+--------------------+------------------+
| 2024-12-31|                48|     Manhattan|        Clinton East|             17.16|
| 2024-12-31|               114|     Manhattan|Greenwich Village...|              63.3|
| 2024-12-31|               186|     Manhattan|Penn Station/Madi...|              21.3|
| 2024-12-31|               249|     Manhattan|        West Village|              32.3|
| 2024-12-31|                68|     Manhattan|        East Chelsea|             23.88|
| 2024-12-31|               142|     Manhattan| Lincoln Square East|             20.52|
| 2024-12-31|                42|     Manhattan|Central Harlem North|              23.3|
| 2024-12-31|               179|        Queens|         Old Astoria|              10.4|
| 2024-12-31|               163|